# PA3 — weights + characteristics, 0CQ single frequency

Holdings-based snapshot from PA Engine as of the most recent calendar quarter end.
Companion to `spar_composite_returns_template.ipynb`, which is returns-based (SPAR).

**Still needed to run:** `PA_DOCUMENT` and the two component names. Account paths, holdings
modes and benchmarks are read off the saved components by Cell 4c.

## One multi-port call per tile

All four strategies go through a single unit per tile — `accounts` is a list, so LC, SMID,
LCS and CONC are calculated together. **2 units total.**

`benchmarks` is left **omitted**, so each account uses its own default benchmark as saved.
That is what makes one unit work across four composites with three different benchmarks:
passing an explicit list would raise the question of how PA pairs 4 accounts to 3
benchmarks — counts don't match, so positional pairing is impossible and the schema doesn't
say whether it cross-products. Omitting the field sidesteps it, and each composite's
benchmark is already correct in the document.

The consequence: **the benchmark is not known from config**, so it is captured from the
response instead (Cell 7). Same for the strategy — a shared unit means rows must be
attributed via the output's account column, so `ACCT_TO_CODE` maps it back and the write
asserts every row resolved. `PER_PAIR_UNITS = True` falls back to one unit per port+bench
pair (8 units) where both come from the unit key, which is easier to debug if attribution
misbehaves.

## Weights: `GROUPSALL`, precalculated at every grain

`GROUPSALL` returns group rows **and** security rows in one response, and **PA supplies
precalculated portfolio, benchmark and active weights at every grain.** Those are
authoritative — never re-derive a sector weight by summing its securities. If a sum
disagrees with the sector row, the sector row is right and the difference is PA's
methodology, not an error to correct.

That is also why the grains are landed as **separate tables** (`pa_sector_weights`,
`pa_security_weights`): both carry their own authoritative weights, so anything summing
across a mixed-grain table double-counts.

Target columns on each holding row:

| Column | Source |
|---|---|
| FSYM perm id | component column (default) |
| **GICS sector** | **derived from the STACH grouping** — see below |
| cash flag | component column, or derived |
| ultimate parent FSYM id | component column, if exposed |
| port / bench / active weight | precalculated by PA at each grain |

### GICS sector is a grouping, not a column

Under `GROUPSALL` the security rows are *nested beneath* their sector's group row, so the
sector must be projected down onto each holding to sit in an adjacent column. Cell 7 uses
the grouping column directly when it is already populated on security rows, otherwise
forward-fills from each preceding group row.

> ⚠️ Forward-fill assumes STACH emits rows in hierarchical order. True of PA's grouped
> output, but an assumption about row order rather than a documented guarantee. Cell 7
> reports which path ran and prints holdings-per-sector counts — check one composite
> against the workstation. A mis-fill mislabels every holding without erroring.

### Benchmark-only securities are dropped at security grain

`HIDE_BENCH_ONLY_SECURITIES = True` drops security rows with no portfolio weight — index
constituents not held. Without it a Russell 3000 benchmark contributes thousands of rows
that no top-N holdings view wants.

> The trade-off: those rows carry the security-level **active** weight of names you *don't*
> own, so dropping them means you cannot show largest underweights-not-held from this
> table. Sector-grain active weight is unaffected. Set the flag False if you need them.
> If the component already hides them the filter is a harmless no-op — the count dropped
> is reported either way.

## Dates: PA resolves `0CQ`, and that answer is authoritative

PA has a `DatesApi`; SPAR does not. `convert_pa_dates_to_absolute_format` turns `0CQ` into
a real `YYYYMMDD`, and **that value is what gets sent and what labels every row** — no
locally computed guess. The SPAR notebook calls the same endpoint so both pipelines agree
on one as-of date.

Note `enddate`, `componentid` and `account` are all **required** on that call (only
`startdate` and `calendar` are optional), so it runs *after* component resolution.

## Metadata is captured, not discarded

Every run records the calculation id, `X-DataDirect-Request-Key`,
`X-FactSet-Api-Request-Key`, rate-limit headers, SDK version, resolved dates, and each
component's id / name / path / currency / snapshot flag into `factset.factset_run_log`.
Request keys are what FactSet support needs to pull the exact request, and they are
worthless if not persisted at the time.

## Fee basis does not apply

Weights and characteristics are holdings attributes — no gross/net distinction.

## Versions

| Package | Version |
|---|---|
| `fds.sdk.PAEngine` | **4.0.0** (upstream latest, 2026-07-21) |
| `fds.sdk.utils` | 3.0.1 |
| `fds.protobuf.stach.extensions` | 1.3.3 |
| `deltalake` | 1.6.2 |

PAEngine 4.0.0 dropped `required` from `PADateParameters.enddate` / `.frequency`, which
**reshuffles positional arguments** across the calculation and dates endpoints
(upstream `BREAKING.md`, 2026-07-21). Every call here passes keywords.

> ⚠️ The PA SDK vendored under `code/python/PAEngine/v3/` in this repo is **2.2.2** and
> predates this. Verify against upstream `main`.

Attach libraries to a **Fabric Environment**, not `%pip`. Interactive first run only:
```
%pip install fds.sdk.PAEngine==4.0.0 fds.sdk.utils==3.0.1 \
             fds.protobuf.stach.extensions==1.3.3 deltalake==1.6.2
```

In [ ]:
# === Cell 1: credentials ===================================================
%run HBCM_Config

In [ ]:
# === Cell 2: imports + API client ==========================================
import json, time, datetime as dt
import pandas as pd

import fds.sdk.PAEngine
from fds.sdk.PAEngine.api import (
    pa_calculations_api, components_api, accounts_api,
    columns_api, groups_api, frequencies_api, dates_api,
)
from fds.sdk.PAEngine.models import (
    PACalculationParametersRoot, PACalculationParameters,
    PAIdentifier, PADateParameters, CalculationMeta,
)
from urllib3 import Retry

SDK_VERSION = fds.sdk.PAEngine.__version__
assert int(SDK_VERSION.split(".")[0]) >= 4, (
    f"fds.sdk.PAEngine {SDK_VERSION} found; this notebook targets >=4.0.0 "
    "(PADateParameters changed shape). Check the bound Fabric Environment."
)
print("PAEngine SDK", SDK_VERSION)

configuration = fds.sdk.PAEngine.Configuration(
    username=FACTSET_USER, password=FACTSET_APIKEY,
)
configuration.retries = Retry(
    total=3, status_forcelist=[500, 502, 503, 504], backoff_factor=2,
    allowed_methods=frozenset(["GET", "POST"]),
)

api_client = fds.sdk.PAEngine.ApiClient(configuration)
calc_api = pa_calculations_api.PACalculationsApi(api_client)
comp_api = components_api.ComponentsApi(api_client)

# --- run metadata ----------------------------------------------------------
# Populated as the notebook proceeds and landed in Cell 9. Request keys are what FactSet
# support needs to retrieve the exact request; they are worthless unless persisted now.
RUN_META = {
    "run_started_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "notebook": "pa_weights_characteristics",
    "engine": "PAEngine",
    "sdk_version": SDK_VERSION,
}

def _field(obj, name):
    """SDK models allow attribute or dict-style access depending on construction."""
    if hasattr(obj, name):
        return getattr(obj, name)
    try:
        return obj.get(name)
    except AttributeError:
        return None

INTERESTING_HEADERS = (
    "X-DataDirect-Request-Key",
    "X-FactSet-Api-Request-Key",
    "X-FactSet-Api-RateLimit-Limit",
    "X-FactSet-Api-RateLimit-Remaining",
    "X-FactSet-Api-RateLimit-Reset",
)

def call_with_headers(fn, *args, **kwargs):
    """Call `fn`'s _with_http_info sibling and return (result, headers).

    The wrapper-returning endpoints don't document their _with_http_info shape, so this
    handles both a 3-tuple and a bare return, and degrades to no headers rather than
    failing the run over telemetry.
    """
    sibling = getattr(fn.__self__, fn.__name__ + "_with_http_info", None)
    if sibling is None:
        return fn(*args, **kwargs), {}
    try:
        out = sibling(*args, **kwargs)
    except TypeError:
        return fn(*args, **kwargs), {}
    if isinstance(out, tuple) and len(out) == 3:
        result, _status, headers = out
        return result, {k: v for k, v in dict(headers or {}).items()
                        if k in INTERESTING_HEADERS}
    return out, {}

In [ ]:
# === Cell 3: THE CONFIG BLOCK ==============================================

CURRENCY = "USD"
AS_OF_RELATIVE = "0CQ"       # resolved to an absolute date by PA in Cell 4b
FREQUENCY = "Single"         # point-in-time snapshot, not a series

# --- the document (the one thing that must be right) -----------------------
PA_DOCUMENT = "<TODO PA3 document path>"

TILES = {
    "weights": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "GROUPSALL",   # group rows AND security rows in one response
        "split_grain": True,
    },
    "characteristics": {
        "component_name": "<TODO exact workstation name>", "pinned_componentid": None,
        "componentdetail": "GROUPS",
        "split_grain": False,
    },
}

# --- accounts --------------------------------------------------------------
# PA needs the HOLDINGS account path, not the returns ACCT the SPAR notebook uses — they
# are different objects, so do not copy one across.
# holdingsmode: B&H, TBR, OMS, EXT or VLT. Cell 4c reads these off the saved component.
STRATEGIES = {
    "LC":   {"label": "Large Cap",           "acct": "<TODO>", "holdingsmode": "B&H"},
    "SMID": {"label": "SMID",                "acct": "<TODO>", "holdingsmode": "B&H"},
    "LCS":  {"label": "Large Cap Select",    "acct": "<TODO>", "holdingsmode": "B&H"},
    "CONC": {"label": "Concentrated Equity", "acct": "<TODO>", "holdingsmode": "B&H"},
}

# --- call shape ------------------------------------------------------------
# False: one unit per tile with all four accounts, `benchmarks` OMITTED so each account
#        uses its saved default benchmark. 2 units. Strategy and benchmark come from the
#        response rather than the unit key.
# True:  one unit per port+bench pair, explicit single benchmark. 8 units. Strategy and
#        benchmark come from the unit key — easier to debug if attribution misbehaves,
#        but needs BENCHMARK_GROUPS filled in.
PER_PAIR_UNITS = False

BENCHMARK_GROUPS = {         # only used when PER_PAIR_UNITS is True
    "r1000": {"label": "Russell 1000", "id": "<TODO>"},   # LC + LCS
    "r2500": {"label": "Russell 2500", "id": "<TODO>"},   # SMID
    "r3000": {"label": "Russell 3000", "id": "<TODO>"},   # CONC
}
BENCH_GROUP_OF = {"LC": "r1000", "LCS": "r1000", "SMID": "r2500", "CONC": "r3000"}

# --- security-grain filtering ---------------------------------------------
# Drop security rows with no portfolio weight — benchmark constituents not held. An R3000
# benchmark otherwise contributes thousands of rows no top-N view wants.
# Cost: loses security-level active weight for names you don't own, so
# largest-underweight-not-held cannot be read off pa_security_weights. Sector grain is
# unaffected. Harmless no-op if the component already hides them.
HIDE_BENCH_ONLY_SECURITIES = True

# --- output column mapping -------------------------------------------------
# STACH labels come from the component, so they can't be known in advance. First candidate
# present wins and is renamed to the canonical key. Cell 7 reports what actually arrived.
COLUMN_HINTS = {
    "fsym_perm_id": ["fsym_perm_id", "fsym_id", "fsym_security_id", "fsym_regional_id",
                     "perm_id"],
    "gics_sector": ["gics_sector", "sector"],       # else derived from the grouping
    "cash_flag": ["cash_flag", "is_cash", "cash", "asset_class", "security_type"],
    "ultimate_parent_fsym_id": ["ultimate_parent_fsym_id", "fsym_ultimate_parent_id",
                                "ult_parent_fsym_id", "ultimate_parent_id"],
    # PA's precalculated weights — authoritative at every grain. Never re-derive.
    "port_weight": ["port_weight", "portfolio_weight", "port._weight", "weight_port",
                    "portfolio_ending_weight"],
    "bench_weight": ["bench_weight", "benchmark_weight", "bench._weight", "weight_bench"],
    "active_weight": ["active_weight", "active_wt", "weight_active", "difference"],
    "account_out": ["portfolio", "account", "portfolio_name", "account_name", "port"],
    "benchmark_out": ["benchmark", "benchmark_name", "bench"],
}
REQUIRED_CANONICAL = ["fsym_perm_id", "port_weight"]   # ultimate parent was "possibly"

# --- OneLake target --------------------------------------------------------
WORKSPACE_ID = "1b9fac18-9d75-4437-ab6c-b6ba44ff46a8"   # HBCM - Production
LAKEHOUSE_ID = "7cdf13b1-4586-4a02-b8ff-72fcf6db1277"   # hbcm_datahub
ONELAKE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
RAW_DIR = f"{ONELAKE}/Files/raw/pa"
TABLE_SECTOR = f"{ONELAKE}/Tables/factset/pa_sector_weights"
TABLE_SECURITY = f"{ONELAKE}/Tables/factset/pa_security_weights"
TABLE_CHARACTERISTICS = f"{ONELAKE}/Tables/factset/pa_characteristics"
TABLE_RUN_LOG = f"{ONELAKE}/Tables/factset/factset_run_log"

ACCT_TO_CODE = {}            # filled once accts are known (Cell 4c -> Cell 3)
print(f"{len(TILES)} tiles, {len(STRATEGIES)} strategies, "
      f"{'per-pair (8 units)' if PER_PAIR_UNITS else 'multi-port (1 unit per tile)'}")

In [ ]:
# === Cell 4: resolve component ids by name — every run ====================
# Ids are not stable: re-saving a component can mint a new one, and a stale id 400s with
# nothing pointing at the id as the cause. The workstation NAME is the contract.

def resolve_component_ids(document=PA_DOCUMENT):
    summary, headers = call_with_headers(comp_api.get_pa_components, document=document)
    RUN_META.setdefault("headers", {}).update(headers)

    by_name = {}
    for cid, meta in (summary.data or {}).items():
        by_name.setdefault(_field(meta, "name"), []).append(cid)

    resolved, drift, missing = {}, [], []
    for tile, cfg in TILES.items():
        hits = by_name.get(cfg["component_name"], [])
        if len(hits) != 1:
            missing.append((tile, cfg["component_name"], len(hits)))
            continue
        resolved[tile] = hits[0]
        if cfg.get("pinned_componentid") and cfg["pinned_componentid"] != hits[0]:
            drift.append((tile, cfg["component_name"], cfg["pinned_componentid"], hits[0]))
    return resolved, drift, missing, by_name

RESOLVED_COMPONENTS, COMPONENT_DRIFT, COMPONENT_MISSING, COMPONENTS_BY_NAME = \
    resolve_component_ids()

for tile, cid in RESOLVED_COMPONENTS.items():
    print(f"{tile:<18} {cid}  ({TILES[tile]['component_name']})")

if COMPONENT_DRIFT:
    print("\n*** COMPONENT ID DRIFT — the component was re-saved. Confirm its columns")
    print("*** still match, then update pinned_componentid:")
    for tile, name, was, now in COMPONENT_DRIFT:
        print(f"    {tile}: {name!r}  {was} -> {now}")

if COMPONENT_MISSING:
    print("\nUnresolved tiles:", COMPONENT_MISSING)
    print("Available component names in", PA_DOCUMENT)
    for name, cids in sorted(COMPONENTS_BY_NAME.items(), key=lambda kv: str(kv[0])):
        print(f"    {name!r}: {cids}")

assert not COMPONENT_MISSING, "every tile needs exactly one matching component name"
RUN_META["components"] = dict(RESOLVED_COMPONENTS)
RUN_META["component_drift"] = [list(d) for d in COMPONENT_DRIFT]

In [ ]:
# === Cell 4b: resolve 0CQ to an absolute date — the authority =============
# PA has a DatesApi; SPAR does not. Whatever this returns is what gets SENT and what LABELS
# every row, in this notebook and in the SPAR one — so there is a single as-of date across
# both pipelines and no locally computed guess in the data.
#
# enddate, componentid and account are all REQUIRED (only startdate and calendar are
# optional), which is why this runs after component resolution.

d_api = dates_api.DatesApi(api_client)

def _fallback_quarter_end(today=None):
    """Last COMPLETED calendar quarter end. Used only if the API call fails."""
    d = today or dt.date.today()
    qe = dt.date(d.year, ((d.month - 1) // 3) * 3 + 1, 1) - dt.timedelta(days=1)
    return qe.strftime("%Y%m%d")

def resolve_as_of(relative=AS_OF_RELATIVE):
    """(absolute YYYYMMDD, source) for `relative`, per PA."""
    probe_component = RESOLVED_COMPONENTS["weights"]
    probe_account = next(iter(STRATEGIES.values()))["acct"]
    try:
        resp, headers = call_with_headers(
            d_api.convert_pa_dates_to_absolute_format,
            enddate=relative, componentid=probe_component, account=probe_account,
            startdate=relative,
        )
        RUN_META.setdefault("headers", {}).update(headers)
        end = _field(resp.data, "enddate")
        start = _field(resp.data, "startdate")
        if end:
            return str(end), f"PA DatesApi (startdate={start})"
    except Exception as e:
        print(f"date conversion failed: {e!r}")
    return _fallback_quarter_end(), "local fallback (PA DatesApi unavailable)"

AS_OF_ABS, AS_OF_SOURCE = resolve_as_of()
AS_OF = AS_OF_ABS            # always send the absolute date — reproducible on replay
asof_tag = AS_OF_ABS

_local = _fallback_quarter_end()
print(f"{AS_OF_RELATIVE} -> {AS_OF_ABS}   source: {AS_OF_SOURCE}")
if AS_OF_ABS != _local:
    # Not necessarily wrong — PA may use a trading-calendar quarter end, or 0CQ may mean
    # the in-progress quarter. But it is worth knowing they differ.
    print(f"NOTE: differs from the locally computed quarter end ({_local}). "
          f"PA's answer is used.")

RUN_META.update({
    "asof_relative": AS_OF_RELATIVE,
    "asof_absolute": AS_OF_ABS,
    "asof_source": AS_OF_SOURCE,
    "asof_local_computed": _local,
    "frequency": FREQUENCY,
    "currency": CURRENCY,
})

# Hand the resolved date to the SPAR notebook so both pipelines share one as-of.
# SPAR reads this if present and falls back to calling PA itself.
try:
    notebookutils.fs.put(f"{ONELAKE}/Files/raw/_asof/{AS_OF_RELATIVE}.json",
                         json.dumps({"relative": AS_OF_RELATIVE, "absolute": AS_OF_ABS,
                                     "source": AS_OF_SOURCE,
                                     "resolved_utc": RUN_META["run_started_utc"]}), True)
except Exception as e:
    print(f"could not publish resolved as-of ({e!r}) — SPAR will resolve it itself")

In [ ]:
# === Cell 4c: read the saved config off each component ====================
# PAComponent exposes the accounts, benchmarks, currency, dates and snapshot flag SAVED IN
# THE DOCUMENT — so the document path plus two component names is enough to fill in account
# paths, holdings modes and benchmark ids. Run once, paste into Cell 3, then skip.

COMPONENT_META = {}
for tile, cid in RESOLVED_COMPONENTS.items():
    try:
        comp = comp_api.get_pa_component_by_id(id=cid)
    except fds.sdk.PAEngine.ApiException as e:
        print(f"{tile}: lookup failed {e.status} {e.body}")
        continue
    d = comp.data
    accts = [{"id": _field(a, "id"), "holdingsmode": _field(a, "holdingsmode")}
             for a in (_field(d, "accounts") or [])]
    benches = [{"id": _field(b, "id"), "holdingsmode": _field(b, "holdingsmode")}
               for b in (_field(d, "benchmarks") or [])]
    COMPONENT_META[tile] = {
        "componentid": cid,
        "name": _field(d, "name"),
        "category": _field(d, "category"),
        "path": _field(d, "path"),
        "currency": _field(d, "currencyisocode"),
        # snapshot=True => point-in-time, which is what a 0CQ weights pull wants. If it is
        # False the component is a subperiod calculation and "Single" may not mean holdings
        # AS OF the quarter end.
        "snapshot": _field(d, "snapshot"),
        "saved_dates": str(_field(d, "dates")),
        "saved_accounts": accts,
        "saved_benchmarks": benches,
    }
    print(f"\n=== {tile} ({COMPONENT_META[tile]['name']}) ===")
    for k in ("path", "category", "currency", "snapshot", "saved_dates"):
        print(f"  {k:<16} {COMPONENT_META[tile][k]}")
    for a in accts:
        print(f"  ACCOUNT          id={a['id']!r} holdingsmode={a['holdingsmode']!r}")
    for b in benches:
        print(f"  BENCHMARK        id={b['id']!r}")

RUN_META["component_meta"] = COMPONENT_META
print("\nMap each ACCOUNT id into Cell 3's STRATEGIES, then set")
print("ACCT_TO_CODE = {s['acct']: c for c, s in STRATEGIES.items()}")

In [ ]:
# === Cell 4d: DISCOVERY — optional lookups ================================
a_api = accounts_api.AccountsApi(api_client)
col_api = columns_api.ColumnsApi(api_client)
grp_api = groups_api.GroupsApi(api_client)
frq_api = frequencies_api.FrequenciesApi(api_client)

# GROUPSALL sets the ROW GRAIN, not which columns exist. If precalculated port/bench/active
# weight, FSYM perm id, cash flag or ultimate parent are missing from the output, the
# component doesn't expose them — edit it, or override PACalculationParameters.columns.
#   print(col_api.get_pa_columns(name="weight", category="", directory=""))
#   print(col_api.get_pa_columns(name="fsym", category="", directory=""))

# Confirm the saved grouping really is GICS sector — Cell 7 projects it onto every holding
# and is blind to what it represents.
#   print(grp_api.get_pa_groups())

#   print(frq_api.get_pa_frequencies())        # confirm "Single"
#   print(a_api.get_accounts(path="Client:/")) # browse holdings accounts
print("uncomment the lookup you need")

In [ ]:
# === Cell 5: build units ==================================================
# Default: one unit per tile, all four accounts, `benchmarks` OMITTED so each account uses
# its saved default. That is what lets four composites with three benchmarks share a unit.

def pa_dates() -> PADateParameters:
    # Keyword args throughout: 4.0.0 dropped `required` on enddate/frequency, which
    # reshuffled positional arguments across the PA calculation endpoints.
    return PADateParameters(startdate=AS_OF, enddate=AS_OF, frequency=FREQUENCY)

def _accounts(codes):
    return [PAIdentifier(id=STRATEGIES[c]["acct"],
                         holdingsmode=STRATEGIES[c]["holdingsmode"]) for c in codes]

def build_multiport_unit(tile_name, tile_cfg) -> PACalculationParameters:
    # `benchmarks` deliberately not passed — omitting it uses each account's default.
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=_accounts(list(STRATEGIES)),
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

def build_pair_unit(tile_name, tile_cfg, code) -> PACalculationParameters:
    bmk = BENCHMARK_GROUPS[BENCH_GROUP_OF[code]]
    return PACalculationParameters(
        componentid=RESOLVED_COMPONENTS[tile_name],
        accounts=_accounts([code]),
        benchmarks=[PAIdentifier(id=bmk["id"])],
        dates=pa_dates(),
        currencyisocode=CURRENCY,
        componentdetail=tile_cfg["componentdetail"],
    )

UNIT_KEYS, units = {}, {}
for tile_name, tile_cfg in TILES.items():
    if PER_PAIR_UNITS:
        for code in STRATEGIES:
            key = f"{tile_name}__{code}"
            units[key] = build_pair_unit(tile_name, tile_cfg, code)
            UNIT_KEYS[key] = (tile_name, code)
    else:
        units[tile_name] = build_multiport_unit(tile_name, tile_cfg)
        UNIT_KEYS[tile_name] = (tile_name, None)   # strategy comes from the response

params_root = PACalculationParametersRoot(
    data=units,
    meta=CalculationMeta(
        contentorganization="SimplifiedRow",
        stach_content_organization="SimplifiedRow",
        contenttype="Json",
        format="JsonStach",
    ),
)
RUN_META["unit_keys"] = list(units)
RUN_META["per_pair_units"] = PER_PAIR_UNITS
for key, (tile_name, code) in UNIT_KEYS.items():
    who = code or ", ".join(STRATEGIES)
    print(f"{key:<26} {TILES[tile_name]['componentdetail']:<10} {who}")

In [ ]:
# === Cell 6: submit + poll ================================================

def run_pa(params_root, deadline=10, poll_interval=3, timeout=900):
    wrapper, headers = call_with_headers(
        calc_api.post_and_calculate,
        x_fact_set_api_long_running_deadline=deadline,
        pa_calculation_parameters_root=params_root,
    )
    RUN_META.setdefault("headers", {}).update(headers)

    code = wrapper.get_status_code()
    if code == 200:
        status_root = wrapper.get_response_200()
    elif code == 201:
        status_root = wrapper.get_response_201()
    elif code == 202:
        status_root = wrapper.get_response_202()
        calc_id = status_root.data.calculationid
        deadline_at = time.time() + timeout
        while True:
            if time.time() > deadline_at:
                calc_api.cancel_calculation_by_id(id=calc_id)
                raise TimeoutError(f"calc {calc_id} exceeded {timeout}s (cancelled)")
            poll = calc_api.get_calculation_status_by_id(id=calc_id)
            if poll.get_status_code() == 200:
                status_root = poll.get_response_200()
                break
            if poll.get_status_code() != 202:
                raise RuntimeError(f"unexpected poll status {poll.get_status_code()}")
            time.sleep(poll_interval)
    else:
        raise RuntimeError(f"unexpected submit status {code}")

    calc_id = status_root.data.calculationid
    out, unit_meta = [], {}
    for unit_id, unit_status in (status_root.data.units or {}).items():
        st = _field(unit_status, "status")
        # Whatever the engine reports per unit — errors, progress, timing — is worth
        # keeping; it is the only record of why a unit came back empty.
        unit_meta[unit_id] = {
            "status": st,
            "error": str(_field(unit_status, "error") or ""),
            "progress": str(_field(unit_status, "progress") or ""),
        }
        if st != "Success":
            out.append((unit_id, None, st))
            continue
        res, rheaders = call_with_headers(
            calc_api.get_calculation_unit_result_by_id, id=calc_id, unit_id=unit_id)
        RUN_META.setdefault("headers", {}).update(rheaders)
        out.append((unit_id, res, st))
    return calc_id, out, unit_meta

calc_id, results, UNIT_META = run_pa(params_root)
RUN_META["calculation_id"] = calc_id
RUN_META["unit_status"] = UNIT_META

failed = [(u, s) for u, r, s in results if r is None]
print(f"calc={calc_id} ok={len(results) - len(failed)} failed={len(failed)}")
for u, s in failed:
    print(f"  FAILED {u}: {s} — {UNIT_META[u]['error']}")
print("\ncaptured headers:")
for k, v in RUN_META.get("headers", {}).items():
    print(f"    {k}: {v}")
assert results and not failed, "resolve failures before writing to the lakehouse"

In [ ]:
# === Cell 7: raw landing, STACH parse, sector projection ==================
from fds.protobuf.stach.extensions.StachExtensionFactory import StachExtensionFactory
from fds.protobuf.stach.extensions.StachVersion import StachVersion

for unit_id, res, _ in results:
    notebookutils.fs.put(f"{RAW_DIR}/asof={asof_tag}/{unit_id}.json",
                         json.dumps(res.to_dict(), default=str), True)
print(f"landed {len(results)} raw payloads under {RAW_DIR}/asof={asof_tag}/")

def stach_to_dataframes(api_response):
    ext = StachExtensionFactory.get_stach_extension(StachVersion.V2)
    return [pd.DataFrame(t.data, columns=t.columns)
            for t in ext.convert(json.dumps(api_response.to_dict(), default=str))]

def norm(cols):
    return [str(c).strip().replace(" ", "_").replace("-", "_").lower() for c in cols]

GROUP_LEVEL_COLS = ("level", "depth", "hierarchy_level", "row_level")
SECURITY_ID_COLS = ("fsym_perm_id", "fsym_id", "security", "security_id", "asset_id",
                    "symbol", "ticker")
GROUPING_COLS = ("gics_sector", "sector", "group", "grouping", "group_name", "group_1")

def first_present(df, candidates):
    return next((c for c in candidates if c in df.columns), None)

def group_mask(df):
    """True on sector/group rows, False on security rows. None if undecidable.

    HEURISTIC — confirm against real output. Both grains carry their own precalculated
    weights, so mixing them double-counts.
    """
    lvl = first_present(df, GROUP_LEVEL_COLS)
    if lvl:
        return pd.to_numeric(df[lvl], errors="coerce").fillna(-1) == 0
    sid = first_present(df, SECURITY_ID_COLS)
    if sid:
        s = df[sid].astype("string").str.strip()
        return s.isna() | (s == "")
    return None

def project_sector(df):
    """Put the GICS sector on every holding row, in an adjacent column.

    Under GROUPSALL the sector is a GROUPING, not a security attribute — security rows sit
    beneath their sector's group row. Use a populated grouping column if there is one,
    otherwise forward-fill from each preceding group row, which assumes STACH emits rows in
    hierarchical order.
    """
    gcol = first_present(df, GROUPING_COLS)
    mask = group_mask(df)
    if gcol is None or mask is None:
        df["gics_sector"] = pd.NA
        return df, "none"
    labels = df[gcol].astype("string").str.strip()
    sec_labels = labels[~mask]
    if len(sec_labels) and (sec_labels.notna() & (sec_labels != "")).all():
        df["gics_sector"] = labels
        return df, f"direct:{gcol}"
    df["gics_sector"] = labels.where(mask).ffill()
    return df, f"ffill:{gcol}"

frames, sector_methods = [], {}
for unit_id, res, _ in results:
    tile_name, code = UNIT_KEYS[unit_id]
    for i, df in enumerate(stach_to_dataframes(res)):
        df = df.copy()
        df.columns = norm(df.columns)
        for canonical, candidates in COLUMN_HINTS.items():
            hit = first_present(df, [c for c in candidates if c != canonical])
            if canonical not in df.columns and hit:
                df = df.rename(columns={hit: canonical})

        if TILES[tile_name]["split_grain"]:
            df, method = project_sector(df)
            sector_methods[f"{unit_id}[{i}]"] = method

        # Strategy: from the unit key in per-pair mode, otherwise from the response's
        # account column, since a multi-port unit holds all four.
        if code is not None:
            df["strategy_code"] = code
        elif "account_out" in df.columns:
            df["strategy_code"] = df["account_out"].astype("string").str.strip() \
                                                   .map(ACCT_TO_CODE)
        else:
            df["strategy_code"] = pd.NA
        df["strategy"] = df["strategy_code"].map(
            {c: s["label"] for c, s in STRATEGIES.items()})
        # Benchmark comes from the response when it wasn't sent.
        df["benchmark_out"] = df["benchmark_out"] if "benchmark_out" in df.columns else pd.NA

        for pos, (col, val) in enumerate([
            ("asof_date",       asof_tag),
            ("asof_relative",   AS_OF_RELATIVE),
            ("tile",            tile_name),
            ("componentid",     RESOLVED_COMPONENTS[tile_name]),
            ("component_name",  TILES[tile_name]["component_name"]),
            ("componentdetail", TILES[tile_name]["componentdetail"]),
            ("currency",        CURRENCY),
            ("calculation_id",  calc_id),
            ("table_ix",        i),
        ]):
            df.insert(pos, col, val)
        frames.append(df)

tidy = pd.concat(frames, ignore_index=True)

print(f"\n{tidy.shape[0]} rows, {tidy.shape[1]} columns")
print("\ncolumns:", list(tidy.columns))
print("\nsector projection method per unit/table:")
for k, v in sector_methods.items():
    print(f"    {k:<30} {v}")

print("\nrequested / required columns:")
for canonical in list(COLUMN_HINTS) + ["gics_sector"]:
    present = canonical in tidy.columns
    filled = int(tidy[canonical].notna().sum()) if present else 0
    flag = "" if present else "   <-- MISSING, add the real name to COLUMN_HINTS"
    print(f"    {canonical:<26} present={present} non_null={filled}{flag}")

print(f"\nstrategy_code resolved: {int(tidy['strategy_code'].notna().sum())} of {len(tidy)}")
if tidy["strategy_code"].isna().any() and not PER_PAIR_UNITS:
    seen = tidy.get("account_out")
    print("  unmapped account values:",
          sorted(set(seen.dropna().unique())) if seen is not None else "(no account column)")
    print("  -> set ACCT_TO_CODE in Cell 3 to map these, or use PER_PAIR_UNITS = True")

_w = tidy[tidy["tile"] == "weights"]
_m = group_mask(_w) if len(_w) else None
print(f"\nweights grain: discriminator_found={_m is not None}"
      + (f" group_rows={int(_m.sum())} security_rows={int((~_m).sum())}"
         if _m is not None else ""))
if _m is not None and "gics_sector" in _w.columns:
    print("\nholdings per sector (sanity-check against the workstation):")
    print(_w[~_m].groupby(["strategy_code", "gics_sector"], dropna=False).size())
display(tidy.head(30))

In [ ]:
# === Cell 8: split by grain, drop benchmark-only securities, write ========
# Guarded rather than commented out: these asserts fail loudly until the grain
# discriminator, column names and strategy attribution are confirmed. A mixed-grain table
# double-counts silently and a sector-less holdings table looks fine.
from deltalake import DeltaTable, write_deltalake

weights = tidy[tidy["tile"] == "weights"].copy()
chars = tidy[tidy["tile"] == "characteristics"].copy()

mask = group_mask(weights)
assert mask is not None, (
    "no grain discriminator found in the GROUPSALL output — inspect Cell 7's column list "
    "and extend GROUP_LEVEL_COLS / SECURITY_ID_COLS"
)
for canonical in REQUIRED_CANONICAL:
    assert canonical in weights.columns, (
        f"{canonical} absent — add its real STACH name to COLUMN_HINTS, or the component "
        f"does not expose it"
    )
assert tidy["strategy_code"].notna().all(), (
    "some rows have no strategy — fill ACCT_TO_CODE from Cell 7's unmapped values, or set "
    "PER_PAIR_UNITS = True so the strategy comes from the unit key"
)

sectors = weights[mask].copy()
securities = weights[~mask].copy()
assert securities["gics_sector"].notna().all(), (
    "some holdings have no GICS sector — the grouping projection did not cover every "
    "security row; check the method Cell 7 reported"
)

# Benchmark-only securities: present in the index, not held. Dropped at security grain
# only — sector grain keeps the full picture, including active weight.
n_before = len(securities)
if HIDE_BENCH_ONLY_SECURITIES:
    pw = pd.to_numeric(securities["port_weight"], errors="coerce")
    securities = securities[pw.notna() & (pw != 0)]
n_dropped = n_before - len(securities)
print(f"security rows: {n_before} -> {len(securities)} "
      f"({n_dropped} benchmark-only dropped, HIDE_BENCH_ONLY_SECURITIES={HIDE_BENCH_ONLY_SECURITIES})")
RUN_META["security_rows_in"] = n_before
RUN_META["security_rows_kept"] = len(securities)
RUN_META["bench_only_dropped"] = n_dropped
RUN_META["sector_rows"] = len(sectors)
RUN_META["sector_methods"] = sector_methods

# PA's precalculated weights are authoritative at BOTH grains. This is a reconciliation
# report, not a correction — if securities do not sum to the sector row, the sector row
# still wins, and the gap is PA methodology (or the dropped benchmark-only names).
if "port_weight" in sectors.columns:
    _s = pd.to_numeric(sectors["port_weight"], errors="coerce").groupby(
        sectors["strategy_code"]).sum()
    _q = pd.to_numeric(securities["port_weight"], errors="coerce").groupby(
        securities["strategy_code"]).sum()
    print("\nport_weight by grain (informational — precalculated values are used as-is):")
    print(pd.DataFrame({"sector_grain": _s, "security_grain": _q}))

def write(path, df, name):
    if df.empty:
        print(f"skip {name}: no rows")
        return
    df = df.astype({c: "string" for c in df.select_dtypes("object").columns})
    try:
        DeltaTable(path).delete(f"asof_date = '{asof_tag}'")
        mode = "append"
    except Exception:
        mode = "overwrite"          # table does not exist yet
    write_deltalake(path, df, mode=mode, schema_mode="merge")
    print(f"wrote {len(df)} rows to {name} (mode={mode})")

write(TABLE_SECTOR, sectors, "factset.pa_sector_weights")
write(TABLE_SECURITY, securities, "factset.pa_security_weights")
write(TABLE_CHARACTERISTICS, chars, "factset.pa_characteristics")

In [ ]:
# === Cell 9: land the run log =============================================
# One row per run. Request keys are the difference between "we'll investigate" and "we know
# exactly what you sent" on a FactSet support ticket, and they only exist at call time.

RUN_META["run_finished_utc"] = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

flat = {"asof_date": asof_tag}
for k, v in RUN_META.items():
    flat[k] = v if isinstance(v, (str, int, float, bool)) or v is None \
        else json.dumps(v, default=str)

run_log = pd.DataFrame([flat]).astype("string")
try:
    DeltaTable(TABLE_RUN_LOG)
    mode = "append"          # append-only: history is the point
except Exception:
    mode = "overwrite"
write_deltalake(TABLE_RUN_LOG, run_log, mode=mode, schema_mode="merge")
print(f"logged run to factset.factset_run_log (mode={mode})")
for k in ("asof_absolute", "asof_source", "calculation_id", "bench_only_dropped"):
    print(f"    {k}: {RUN_META.get(k)}")

# Re-frame any Direct Lake semantic model BEFORE running VACUUM — vacuuming files a framed
# model still points at gives query errors on missing files. Order: write -> frame -> vacuum.

## To finish

1. `PA_DOCUMENT` + the two `component_name`s (Cell 3).
2. Run Cell 4c — it prints the accounts, holdings modes and benchmarks saved in each
   component. Map them into `STRATEGIES`, then set
   `ACCT_TO_CODE = {s["acct"]: c for c, s in STRATEGIES.items()}`.
3. Run to Cell 7 and read its reports: which columns arrived, which sector-projection path
   ran, how many rows resolved to a strategy, and holdings-per-sector counts.
4. Add real column names to `COLUMN_HINTS` until Cell 8's asserts pass.

If `ACCT_TO_CODE` can't be made to match the output's account values, set
`PER_PAIR_UNITS = True` — 8 units instead of 2, but strategy and benchmark come from the
unit key and nothing has to be matched.

## What to verify before trusting the numbers

- **The grain discriminator.** `group_mask()` guesses at a level/depth column, then a null
  security id. Both grains carry precalculated weights, so a wrong split double-counts.
- **The sector projection.** Cell 7 reports `direct:` or `ffill:`. `ffill` relies on STACH
  row order being hierarchical — cross-check holdings-per-sector against the workstation.
- **Active weight exists at both grains.** `GROUPSALL` sets the row grain, not the column
  set. If the component exposes only portfolio and benchmark weight, edit it or override
  `PACalculationParameters.columns`.
- **`snapshot` is True on the weights component** (Cell 4c). If False, it is a subperiod
  calculation and `Single` at `0CQ` may not mean holdings *as of* the quarter end.
- **The grouping really is GICS sector**, not another saved scheme (`get_pa_groups()`).
- **Cash.** Check whether cash lands inside a sector, in its own group, or with no sector,
  and whether `cash_flag` distinguishes it. Cash has no portfolio weight in some
  configurations, in which case `HIDE_BENCH_ONLY_SECURITIES` would drop it — the row
  counts printed in Cell 8 are the place to notice that.
- **PA's `0CQ`** may differ from the naive last-completed-quarter-end if it uses a trading
  calendar. Cell 4b prints both and prefers PA's.

## Sources

Verified against **upstream `FactSet/enterprise-sdk` `main`** (PAEngine v3, SDK 4.0.0) —
`PACalculationParameters.md` (`accounts`/`benchmarks` as lists, `componentdetail` values,
`columns`, `groups`), `PAComponent.md` (saved `accounts`, `benchmarks`, `dates`,
`snapshot`), `PADateParameters.md`, `PAIdentifier.md`, `PACalculationsApi.md`,
`ComponentsApi.md`, `DatesApi.md` (`enddate` + `componentid` + `account` required;
`startdate`/`calendar` optional), `DateParametersSummary.md`, `ColumnsApi.md`,
`GroupsApi.md`, `FrequenciesApi.md`, and `BREAKING.md`.

Not against `code/python/PAEngine/v3/` in this repo, which is pinned at 2.2.2.